[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/MMAudio_Colab.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/MMAudio_Colab.ipynb)

# 🎬🔊 MMAudio (`large_44k_v2`) — Video-to-Audio & Text-to-Audio Synchronized Foley Synthesis (CVPR 2025, MIT)

> **Runtime required:** GPU — **T4 (16 GB)**, **L4 (24 GB)**, or **A100 (40/80 GB)**. Runs in `bfloat16` using ~6.5 GB VRAM on a free Colab T4 GPU and synthesizes **8 seconds of frame-synchronized 44.1 kHz audio in ~3–6 seconds**!

A complete Google Colab studio for **[MMAudio](https://github.com/hkchengrex/MMAudio)** (`hkchengrex/MMAudio`, **CVPR 2025**, **MIT License**) — a multimodal joint Video + Text + Audio conditional flow-matching transformer that generates **frame-accurate synchronized Foley sound effects and ambient audio** directly from silent videos (`Synchformer` 25 FPS temporal alignment + `CLIP` 8 FPS semantic conditioning) or standalone text prompts.

**Upstream links:** [hkchengrex/MMAudio](https://github.com/hkchengrex/MMAudio) · [hkchengrex/MMAudio (HF)](https://huggingface.co/hkchengrex/MMAudio) · [Project Page](https://hkchengrex.com/MMAudio/) · [arXiv 2412.15322](https://arxiv.org/abs/2412.15322) · [MIT License](https://github.com/hkchengrex/MMAudio/blob/main/LICENSE)

---

## 🧩 Model Variants (`mmaudio.eval_utils.all_model_cfg`)

| Variant | Sample Rate | Params | Recommended Use | Peak VRAM (`bf16`) |
|---|---|---|---|---|
| **`large_44k_v2` (Default)** | **44.1 kHz** | 1.03B | Highest-fidelity Video-to-Audio Foley & Text-to-Audio SFX (improved v2 training) | ~6.5 GB |
| **`large_44k`** | 44.1 kHz | 1.03B | Original CVPR 2025 large 44.1 kHz checkpoint | ~6.5 GB |
| **`medium_44k`** | 44.1 kHz | 621M | Balanced speed & 44.1 kHz fidelity | ~5.0 GB |
| **`small_44k`** | 44.1 kHz | 157M | Ultra-fast 44.1 kHz synthesis | ~3.5 GB |
| **`small_16k`** | 16.0 kHz | 157M | Lightweight 16 kHz BigVGAN vocoder variant | ~3.2 GB |

## 🔗 Cross-Notebook Video-to-Foley Pipeline (`Wan2.2` / `LTX-2-5` / `MiniMax-H3` $\rightarrow$ `MMAudio`)

- Includes a **📂 Cross-Notebook Drive Video Picker** that automatically discovers silent `.mp4` videos generated by **`Wan2.2_Colab.ipynb`**, **`Wan2.2_Animate_Colab.ipynb`**, **`LTX-2-5_ComfyUI_Colab.ipynb`**, and **`MiniMax-H3_Colab.ipynb`** inside `/content/drive/MyDrive/AEI_3D_Out/`.
- Outputs **both** a frame-synchronized `.mp4` video (muxed via `make_video`) **and** a standalone 44.1 kHz `.wav` audio file saved to `/content/drive/MyDrive/AEI_TTS_Out/MMAudio/` and `/content/drive/MyDrive/AEI_3D_Out/MMAudio/`.

In [ ]:
# @title STEP 1 — Clone hkchengrex/MMAudio & Install Dependencies
# @markdown Mounts Google Drive, configures `HF_HOME` at `/content/drive/MyDrive/AEI_TTS_Cache/MMAudio`, clones `https://github.com/hkchengrex/MMAudio.git`, links `weights/` and `ext_weights/` to persistent Google Drive storage, and installs runtime dependencies (`torchdiffeq`, `open_clip_torch`, `timm`, `av`, `nitrous-ema`, `tensordict`, `gradio>=5.0.0`, `tqdm`).

import os, sys, subprocess, shutil, time
import torch

print("=" * 72)
print("🎬🔊 MMAudio Studio — Environment Setup & Installation")
print("=" * 72)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[GPU] PyTorch {torch.__version__} · Device = {DEVICE}", flush=True)
if DEVICE == "cuda":
    try:
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"[GPU] {gpu_name} ({vram_gb:.1f} GB VRAM)", flush=True)
    except Exception as e:
        print(f"[GPU] Could not query GPU: {e}", flush=True)

MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
if MOUNT_GOOGLE_DRIVE:
    try:
        if not os.path.exists("/content/drive/MyDrive"):
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"[Drive] Notice — running with local storage fallback ({e})", flush=True)

if os.path.exists("/content/drive/MyDrive"):
    CACHE_ROOT = "/content/drive/MyDrive/AEI_TTS_Cache/MMAudio"
    OUT_DIR_TTS = "/content/drive/MyDrive/AEI_TTS_Out/MMAudio"
    OUT_DIR_3D = "/content/drive/MyDrive/AEI_3D_Out/MMAudio"
else:
    CACHE_ROOT = "/content/cache/MMAudio"
    OUT_DIR_TTS = "/content/outputs/MMAudio_TTS"
    OUT_DIR_3D = "/content/outputs/MMAudio_3D"

for d in (CACHE_ROOT, OUT_DIR_TTS, OUT_DIR_3D, "/content/mmaudio_out"):
    os.makedirs(d, exist_ok=True)

os.environ.setdefault("HF_HOME", CACHE_ROOT)
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(CACHE_ROOT, "hub")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print(f"[Cache] HF_HOME      = {CACHE_ROOT}", flush=True)
print(f"[Out 1] OUT_DIR_TTS  = {OUT_DIR_TTS}", flush=True)
print(f"[Out 2] OUT_DIR_3D   = {OUT_DIR_3D}", flush=True)

if not shutil.which("ffmpeg"):
    print("[apt] Installing ffmpeg...", flush=True)
    subprocess.run(["apt-get", "-qq", "-y", "install", "ffmpeg"], check=False)

REPO_DIR = "/content/MMAudio"
if not os.path.exists(os.path.join(REPO_DIR, "mmaudio", "eval_utils.py")):
    print("\n[git] Cloning https://github.com/hkchengrex/MMAudio.git ...", flush=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/hkchengrex/MMAudio.git", REPO_DIR], check=True)
else:
    print(f"\n[git] Repository already present at {REPO_DIR}", flush=True)

# Symlink /content/MMAudio/weights and /content/MMAudio/ext_weights -> Drive CACHE_ROOT
for sub in ("weights", "ext_weights"):
    drive_sub = os.path.join(CACHE_ROOT, sub)
    os.makedirs(drive_sub, exist_ok=True)
    repo_sub = os.path.join(REPO_DIR, sub)
    if os.path.islink(repo_sub):
        pass
    elif os.path.isdir(repo_sub):
        for fn in os.listdir(repo_sub):
            sp = os.path.join(repo_sub, fn)
            dp = os.path.join(drive_sub, fn)
            if not os.path.exists(dp):
                shutil.move(sp, dp)
        shutil.rmtree(repo_sub, ignore_errors=True)
        os.symlink(drive_sub, repo_sub)
    else:
        os.symlink(drive_sub, repo_sub)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

t0 = time.time()
print("\n[pip] Installing MMAudio runtime dependencies...", flush=True)
PKGS = [
    "torchdiffeq>=0.2.5",
    "open_clip_torch>=2.29.0",
    "timm>=1.0.12",
    "av>=14.0.1",
    "nitrous-ema",
    "hydra-core>=1.3.2",
    "hydra_colorlog",
    "tensordict>=0.6.1",
    "colorlog",
    "einops>=0.6.0",
    "python-dotenv",
    "librosa>=0.10.2",
    "soundfile",
    "scipy>=1.11.0",
    "huggingface_hub>=0.26.0",
    "tqdm>=4.66.0",
    "gradio>=5.0.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PKGS, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO_DIR], check=True)
print(f"[pip] Installed in {time.time() - t0:.1f}s", flush=True)

print("\n[Verify] Checking MMAudio imports...", flush=True)
try:
    from mmaudio.eval_utils import ModelConfig, VideoInfo, all_model_cfg, generate, load_video, make_video
    from mmaudio.model.flow_matching import FlowMatching
    from mmaudio.model.networks import MMAudio, get_my_mmaudio
    from mmaudio.model.utils.features_utils import FeaturesUtils
    print(f"  ✓ MMAudio imported OK! Variants available: {list(all_model_cfg.keys())}")
except Exception as e:
    print(f"  ✗ Import failed: {e}")
    raise

print("\n[Done] STEP 1 complete — proceed to STEP 2.")

In [ ]:
# @title STEP 2 — Pre-Cache MMAudio (`large_44k_v2`), Synchformer & VAE Weights
# @markdown Downloads the `large_44k_v2` model weights (`mmaudio_large_44k_v2.pth`), 44.1 kHz Audio VAE (`v1-44.pth`), and `Synchformer` visual-temporal encoder (`synchformer_state_dict.pth`) into `/content/drive/MyDrive/AEI_TTS_Cache/MMAudio/` via `model.download_if_needed()`. Also creates a sample silent test video if none is present yet.

import os, sys, time, subprocess
from pathlib import Path
from tqdm.auto import tqdm

REPO_DIR = "/content/MMAudio"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

CACHE_ROOT = os.environ.get("HF_HOME", "/content/drive/MyDrive/AEI_TTS_Cache/MMAudio")
os.environ.setdefault("HF_HOME", CACHE_ROOT)
os.environ["HF_HOME"] = CACHE_ROOT

from mmaudio.eval_utils import all_model_cfg

VARIANT_TO_CACHE = "large_44k_v2"  # @param ["large_44k_v2", "large_44k", "medium_44k", "small_44k", "small_16k"]

print(f"[Cache] Downloading MMAudio variant '{VARIANT_TO_CACHE}' + VAE + Synchformer to {CACHE_ROOT} ...", flush=True)
t0 = time.time()
model_cfg = all_model_cfg[VARIANT_TO_CACHE]
model_cfg.download_if_needed()

for label, p in [
    ("MMAudio Flow-Matching Net", model_cfg.model_path),
    ("Audio VAE / Vocoder", model_cfg.vae_path),
    ("Synchformer Video Encoder", model_cfg.synchformer_ckpt),
]:
    if p and os.path.exists(p):
        mb = os.path.getsize(p) / (1024 ** 2)
        print(f"  ✓ {label:<28}: {p} ({mb:.1f} MB)")

# Create a 5-second sample silent test video using ffmpeg testsrc2 if no sample exists
SAMPLE_VIDEO_PATH = os.path.join(CACHE_ROOT, "sample_silent_clip.mp4")
if not os.path.exists(SAMPLE_VIDEO_PATH):
    try:
        subprocess.run(
            [
                "ffmpeg", "-y", "-f", "lavfi", "-i", "testsrc2=size=640x360:rate=25:duration=5",
                "-c:v", "libx264", "-pix_fmt", "yuv420p", SAMPLE_VIDEO_PATH,
            ],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True,
        )
        print(f"  ✓ Created 5s sample test video at {SAMPLE_VIDEO_PATH}")
    except Exception as e:
        print(f"  ⚠ Could not create sample video: {e}")

print(f"\n[Done] STEP 2 complete in {time.time() - t0:.1f}s — proceed to STEP 3.")

In [ ]:
# @title STEP 3 — Core MMAudio Video-to-Audio & Text-to-Audio Synthesis Engine
# @markdown Defines the lazy `MMAudio` + `FeaturesUtils` loader and high-level `synthesize_video_foley()` and `synthesize_text_sfx()` functions, saving both synchronized `.mp4` videos and 44.1 kHz `.wav` audio files to `/content/drive/MyDrive/AEI_TTS_Out/MMAudio/` and `/content/drive/MyDrive/AEI_3D_Out/MMAudio/`.

import os, sys, gc, time, shutil, random, builtins
from pathlib import Path
from fractions import Fraction
from typing import Optional, Dict, List
import numpy as np
import torch
import soundfile as sf
from tqdm.auto import tqdm

REPO_DIR = "/content/MMAudio"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from mmaudio.eval_utils import (
    ModelConfig,
    VideoInfo,
    all_model_cfg,
    generate,
    load_image,
    load_video,
    make_video,
    setup_eval_logging,
)
from mmaudio.model.flow_matching import FlowMatching
from mmaudio.model.networks import MMAudio, get_my_mmaudio
from mmaudio.model.sequence_config import SequenceConfig
from mmaudio.model.utils.features_utils import FeaturesUtils

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_ROOT = os.environ.get("HF_HOME", "/content/drive/MyDrive/AEI_TTS_Cache/MMAudio")
os.environ.setdefault("HF_HOME", CACHE_ROOT)

if os.path.exists("/content/drive/MyDrive"):
    OUT_DIR_TTS = "/content/drive/MyDrive/AEI_TTS_Out/MMAudio"
    OUT_DIR_3D = "/content/drive/MyDrive/AEI_3D_Out/MMAudio"
else:
    OUT_DIR_TTS = "/content/outputs/MMAudio_TTS"
    OUT_DIR_3D = "/content/outputs/MMAudio_3D"

os.makedirs(OUT_DIR_TTS, exist_ok=True)
os.makedirs(OUT_DIR_3D, exist_ok=True)
os.makedirs("/content/mmaudio_out", exist_ok=True)

DEFAULT_VARIANT = "large_44k_v2"  # @param ["large_44k_v2", "large_44k", "medium_44k", "small_44k", "small_16k"]
FULL_PRECISION_FP32 = False  # @param {type:"boolean"}

_MMAUDIO_STATE: Dict[str, object] = {
    "variant": None,
    "dtype": None,
    "net": None,
    "feature_utils": None,
    "seq_cfg": None,
    "model_cfg": None,
}


def free_mmaudio_vram() -> str:
    for k in ("net", "feature_utils", "seq_cfg", "model_cfg"):
        _MMAUDIO_STATE[k] = None
    _MMAUDIO_STATE["variant"] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_b, tot_b = torch.cuda.mem_get_info(0)
        return f"MMAudio unloaded · GPU free: {free_b/1e9:.1f}/{tot_b/1e9:.1f} GB"
    return "MMAudio unloaded (CPU)"


def get_mmaudio_bundle(variant: str = DEFAULT_VARIANT, full_precision: bool = FULL_PRECISION_FP32):
    os.chdir(REPO_DIR)
    dtype = torch.float32 if (full_precision or DEVICE == "cpu") else torch.bfloat16
    if (
        _MMAUDIO_STATE["net"] is not None
        and _MMAUDIO_STATE["variant"] == variant
        and _MMAUDIO_STATE["dtype"] == dtype
    ):
        return _MMAUDIO_STATE["net"], _MMAUDIO_STATE["feature_utils"], _MMAUDIO_STATE["seq_cfg"], _MMAUDIO_STATE["model_cfg"]

    if _MMAUDIO_STATE["net"] is not None:
        free_mmaudio_vram()

    if variant not in all_model_cfg:
        raise ValueError(f"Unknown MMAudio variant '{variant}'. Available: {list(all_model_cfg.keys())}")

    t0 = time.time()
    print(f"[MMAudio] Loading variant='{variant}' (dtype={dtype}) on {DEVICE} ...", flush=True)
    model_cfg: ModelConfig = all_model_cfg[variant]
    model_cfg.download_if_needed()
    seq_cfg: SequenceConfig = model_cfg.seq_cfg

    net: MMAudio = get_my_mmaudio(model_cfg.model_name).to(DEVICE, dtype).eval()
    net.load_weights(torch.load(model_cfg.model_path, map_location=DEVICE, weights_only=True))

    feature_utils = FeaturesUtils(
        tod_vae_ckpt=model_cfg.vae_path,
        synchformer_ckpt=model_cfg.synchformer_ckpt,
        enable_conditions=True,
        mode=model_cfg.mode,
        bigvgan_vocoder_ckpt=model_cfg.bigvgan_16k_path,
        need_vae_encoder=False,
    ).to(DEVICE, dtype).eval()

    _MMAUDIO_STATE.update({
        "variant": variant,
        "dtype": dtype,
        "net": net,
        "feature_utils": feature_utils,
        "seq_cfg": seq_cfg,
        "model_cfg": model_cfg,
    })
    print(f"[MMAudio] Ready in {time.time() - t0:.1f}s (sampling_rate={seq_cfg.sampling_rate} Hz)", flush=True)
    return net, feature_utils, seq_cfg, model_cfg


def discover_aei_drive_videos(max_files: int = 60) -> List[str]:
    """Scan AEI video generation folders (Wan2.2, LTX-2-5, MiniMax-H3, InfiniteTalk, etc.) for .mp4/.mov clips."""
    search_roots = [
        OUT_DIR_3D,
        "/content/drive/MyDrive/AEI_3D_Out",
        "/content/drive/MyDrive/AEI_TTS_Out",
        CACHE_ROOT,
    ]
    exts = {".mp4", ".mov", ".webm", ".mkv", ".avi"}
    found, seen = [], set()
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dp, _, fs in os.walk(root):
            for fn in fs:
                if os.path.splitext(fn)[1].lower() in exts:
                    p = os.path.abspath(os.path.join(dp, fn))
                    if p not in seen:
                        seen.add(p)
                        try:
                            found.append((os.path.getmtime(p), p))
                        except OSError:
                            pass
    found.sort(key=lambda x: x[0], reverse=True)
    return [p for _, p in found[:max_files]]


def _save_stereo_wav_and_mirror(audio_tensor: torch.Tensor, sr: int, base_stem: str) -> Tuple[str, str, np.ndarray]:
    """Save 44.1 kHz stereo WAV to both AEI_TTS_Out/MMAudio and AEI_3D_Out/MMAudio."""
    audio_np = audio_tensor.detach().float().cpu().squeeze().numpy()
    audio_np = np.clip(audio_np, -1.0, 1.0)
    # Duplicate mono to 2-channel stereo for universal NLE / avatar compatibility
    if audio_np.ndim == 1:
        stereo_np = np.stack([audio_np, audio_np], axis=-1)
    else:
        stereo_np = audio_np.T if audio_np.shape[0] <= 2 else audio_np

    wav_tts = os.path.join(OUT_DIR_TTS, f"{base_stem}.wav")
    wav_3d = os.path.join(OUT_DIR_3D, f"{base_stem}.wav")
    sf.write(wav_tts, stereo_np, int(sr), subtype="PCM_16")
    if os.path.abspath(wav_tts) != os.path.abspath(wav_3d):
        try:
            shutil.copy2(wav_tts, wav_3d)
        except Exception:
            pass
    return wav_tts, wav_3d, stereo_np


@torch.inference_mode()
def synthesize_video_foley(
    video_path: str,
    prompt: str = "",
    negative_prompt: str = "music, speech, muffled, low quality",
    duration: float = 8.0,
    cfg_strength: float = 4.5,
    num_steps: int = 25,
    seed: int = 42,
    mask_away_clip: bool = False,
    variant: str = DEFAULT_VARIANT,
    prefix: str = "mmaudio_v2a",
) -> Dict[str, object]:
    """Generate frame-synchronized Foley audio for a video (.mp4 -> muxed .mp4 + 44.1kHz stereo .wav)."""
    if not video_path or not os.path.exists(str(video_path)):
        raise FileNotFoundError(f"Video file not found: {video_path}")

    net, feature_utils, seq_cfg, _ = get_mmaudio_bundle(variant=variant)
    actual_seed = int(seed) if (seed is not None and int(seed) >= 0) else random.randint(1, 2**31 - 1)
    rng = torch.Generator(device=DEVICE).manual_seed(actual_seed)
    fm = FlowMatching(min_sigma=0, inference_mode="euler", num_steps=int(num_steps))

    t0 = time.time()
    video_info = load_video(Path(video_path), float(duration))
    clip_frames = None if mask_away_clip else video_info.clip_frames.unsqueeze(0)
    sync_frames = video_info.sync_frames.unsqueeze(0)
    eff_duration = float(video_info.duration_sec)

    seq_cfg.duration = eff_duration
    net.update_seq_lengths(seq_cfg.latent_seq_len, seq_cfg.clip_seq_len, seq_cfg.sync_seq_len)

    audios = generate(
        clip_frames,
        sync_frames,
        [prompt or ""],
        negative_text=[negative_prompt or ""],
        feature_utils=feature_utils,
        net=net,
        fm=fm,
        rng=rng,
        cfg_strength=float(cfg_strength),
    )
    audio_cpu = audios.float().cpu()[0]
    sr = int(seq_cfg.sampling_rate)

    ts = int(time.time())
    vid_stem = Path(video_path).stem[:24]
    base_name = f"{prefix}_{vid_stem}_seed{actual_seed}_{ts}"

    wav_tts, wav_3d, stereo_np = _save_stereo_wav_and_mirror(audio_cpu, sr, base_name)
    mp4_3d = os.path.join(OUT_DIR_3D, f"{base_name}.mp4")
    mp4_tts = os.path.join(OUT_DIR_TTS, f"{base_name}.mp4")

    make_video(video_info, Path(mp4_3d), audio_cpu, sampling_rate=sr)
    if os.path.exists(mp4_3d) and os.path.abspath(mp4_3d) != os.path.abspath(mp4_tts):
        try:
            shutil.copy2(mp4_3d, mp4_tts)
        except Exception:
            pass

    elapsed = time.time() - t0
    gc.collect()
    return {
        "video_path": mp4_3d,
        "video_path_tts": mp4_tts,
        "wav_path": wav_tts,
        "wav_path_3d": wav_3d,
        "audio_np": stereo_np,
        "sr": sr,
        "duration_s": eff_duration,
        "elapsed_s": elapsed,
        "seed_used": actual_seed,
        "variant": variant,
    }


@torch.inference_mode()
def synthesize_text_sfx(
    prompt: str,
    negative_prompt: str = "music, muffled, distortion",
    duration: float = 8.0,
    cfg_strength: float = 4.5,
    num_steps: int = 25,
    seed: int = 42,
    variant: str = DEFAULT_VARIANT,
    prefix: str = "mmaudio_t2a",
) -> Dict[str, object]:
    """Generate standalone 44.1 kHz stereo sound effects / ambience from a text prompt alone."""
    if not prompt or not prompt.strip():
        raise ValueError("Provide a non-empty text prompt for Text-to-Audio synthesis.")

    net, feature_utils, seq_cfg, _ = get_mmaudio_bundle(variant=variant)
    actual_seed = int(seed) if (seed is not None and int(seed) >= 0) else random.randint(1, 2**31 - 1)
    rng = torch.Generator(device=DEVICE).manual_seed(actual_seed)
    fm = FlowMatching(min_sigma=0, inference_mode="euler", num_steps=int(num_steps))

    t0 = time.time()
    seq_cfg.duration = float(duration)
    net.update_seq_lengths(seq_cfg.latent_seq_len, seq_cfg.clip_seq_len, seq_cfg.sync_seq_len)

    audios = generate(
        None,
        None,
        [prompt.strip()],
        negative_text=[negative_prompt or ""],
        feature_utils=feature_utils,
        net=net,
        fm=fm,
        rng=rng,
        cfg_strength=float(cfg_strength),
    )
    audio_cpu = audios.float().cpu()[0]
    sr = int(seq_cfg.sampling_rate)

    slug = "".join(c if c.isalnum() else "_" for c in prompt.strip()[:24]).strip("_") or "sfx"
    base_name = f"{prefix}_{slug}_d{int(duration)}s_seed{actual_seed}_{int(time.time())}"
    wav_tts, wav_3d, stereo_np = _save_stereo_wav_and_mirror(audio_cpu, sr, base_name)
    elapsed = time.time() - t0
    gc.collect()
    return {
        "wav_path": wav_tts,
        "wav_path_3d": wav_3d,
        "audio_np": stereo_np,
        "sr": sr,
        "duration_s": float(duration),
        "elapsed_s": elapsed,
        "seed_used": actual_seed,
        "variant": variant,
    }


builtins.synthesize_video_foley = synthesize_video_foley
builtins.synthesize_text_sfx = synthesize_text_sfx
builtins.discover_aei_drive_videos = discover_aei_drive_videos
builtins.free_mmaudio_vram = free_mmaudio_vram
builtins.MMAUDIO_OUT_TTS = OUT_DIR_TTS
builtins.MMAUDIO_OUT_3D = OUT_DIR_3D

vids = discover_aei_drive_videos()
print(f"[Core] STEP 3 ready. Found {len(vids)} candidate video(s) across AEI Drive folders.")
print(f"       Outputs saved to: {OUT_DIR_TTS} AND {OUT_DIR_3D}")

In [ ]:
# @title STEP 4 — Launch Gradio UI (Video-to-Audio Synchronized Foley + Text-to-Audio SFX Studio)
# @markdown Launches the interactive MMAudio Gradio application:
# @markdown 1. **Tab 1: Video-to-Audio Synchronized Foley (`.mp4` $\rightarrow$ `.mp4` + 44.1 kHz Stereo `.wav`)** with **📂 Cross-Notebook Drive Video Picker** (`Wan2.2`, `LTX-2-5`, `MiniMax-H3`, etc.)
# @markdown 2. **Tab 2: Text-to-Audio Sound Effects & Ambience (`.wav`)** with curated SFX prompt presets

import os, time
import gradio as gr
from IPython.display import clear_output

init_videos = discover_aei_drive_videos(max_files=60)
VID_CHOICES = ["(Use uploaded video above)"] + init_videos

SFX_PRESETS = {
    "Rainstorm & Distant Thunder": "Heavy rain falling on pavement, water splashing in puddles, and deep distant rolling thunder",
    "Footsteps on Gravel": "Crisp footsteps walking steadily on crunchy gravel path outdoors with subtle wind",
    "Sci-Fi Spaceship Cockpit": "Futuristic spaceship cockpit hum, computer console beeps, and hydraulic door hiss",
    "Ocean Waves on Rocky Shore": "Ocean waves crashing against coastal rocks, fizzing sea foam, and seagulls calling in the distance",
    "Cozy Fireplace & Crackle": "Warm wood fireplace crackling, popping embers, and soft indoor room ambience",
    "Cyberpunk Neon Street": "Busy rainy cyberpunk night street, passing electric vehicles on wet asphalt, neon buzz",
    "Sword Clash & Armor Foley": "Metallic medieval sword clashing, steel blade ring, leather armor movement",
}


def _resolve_video(up_vid, dr_vid):
    if up_vid and os.path.exists(str(up_vid)):
        return str(up_vid)
    if dr_vid and dr_vid != "(Use uploaded video above)" and os.path.exists(str(dr_vid)):
        return str(dr_vid)
    return None


def _refresh_videos():
    v = ["(Use uploaded video above)"] + discover_aei_drive_videos(max_files=60)
    return gr.update(choices=v, value=v[1] if len(v) > 1 else v[0])


def _ui_v2a(vid_up, vid_dr, prompt, neg_prompt, duration, cfg_strength, steps, seed, mask_clip, variant, progress=gr.Progress()):
    vid_path = _resolve_video(vid_up, vid_dr)
    if not vid_path:
        raise gr.Error("Upload a video file or select one from the Cross-Notebook Drive Video Picker.")
    try:
        progress(0.15, desc=f"Synthesizing synchronized Foley ({variant})...")
        res = synthesize_video_foley(
            video_path=vid_path,
            prompt=prompt,
            negative_prompt=neg_prompt,
            duration=float(duration),
            cfg_strength=float(cfg_strength),
            num_steps=int(steps),
            seed=int(seed),
            mask_away_clip=bool(mask_clip),
            variant=variant,
            prefix="mmaudio_v2a",
        )
        progress(1.0, desc="Done!")
        info = (
            f"Variant: {res['variant']} ({res['sr']} Hz Stereo) · Duration: {res['duration_s']:.2f}s · Wall: {res['elapsed_s']:.1f}s · Seed: {res['seed_used']}\n"
            f"Source Video : {os.path.basename(vid_path)}\n"
            f"Muxed MP4    : {res['video_path']}\n"
            f"Stereo WAV   : {res['wav_path']}"
        )
        return res["video_path"], (res["sr"], res["audio_np"]), res["video_path"], res["wav_path"], info
    except Exception as e:
        raise gr.Error(f"Video-to-Audio Foley synthesis failed: {e}")


def _ui_t2a(prompt, neg_prompt, duration, cfg_strength, steps, seed, variant, progress=gr.Progress()):
    if not prompt or not prompt.strip():
        raise gr.Error("Enter a sound effect description in the Prompt box.")
    try:
        progress(0.15, desc=f"Synthesizing Text-to-Audio SFX ({variant})...")
        res = synthesize_text_sfx(
            prompt=prompt,
            negative_prompt=neg_prompt,
            duration=float(duration),
            cfg_strength=float(cfg_strength),
            num_steps=int(steps),
            seed=int(seed),
            variant=variant,
            prefix="mmaudio_t2a",
        )
        progress(1.0, desc="Done!")
        info = (
            f"Variant: {res['variant']} ({res['sr']} Hz Stereo) · Duration: {res['duration_s']:.2f}s · Wall: {res['elapsed_s']:.1f}s · Seed: {res['seed_used']}\n"
            f"Saved to AEI_TTS_Out: {res['wav_path']}\n"
            f"Saved to AEI_3D_Out : {res['wav_path_3d']}"
        )
        return (res["sr"], res["audio_np"]), res["wav_path"], info
    except Exception as e:
        raise gr.Error(f"Text-to-Audio SFX synthesis failed: {e}")


with gr.Blocks(title="MMAudio — Synchronized Video-to-Audio Foley & Text-to-Audio SFX", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 🎬🔊 MMAudio (`large_44k_v2`) — Synchronized Video-to-Audio Foley & Text-to-Audio SFX Studio\n"
        "CVPR 2025 · MIT License · Frame-Accurate 25 FPS Synchformer Alignment · 44.1 kHz Stereo Output"
    )
    welcome_banner = gr.Markdown()

    with gr.Tabs():
        # ── TAB 1: Video-to-Audio Synchronized Foley ──
        with gr.Tab("1. Video-to-Audio Synchronized Foley (.mp4 -> .mp4 + .wav)"):
            with gr.Row():
                with gr.Column(scale=3):
                    v1_up = gr.Video(label="Upload Silent or Source Video (.mp4 / .mov / .webm)")
                    with gr.Row():
                        v1_dr = gr.Dropdown(
                            choices=VID_CHOICES,
                            value=VID_CHOICES[1] if len(VID_CHOICES) > 1 else VID_CHOICES[0],
                            label="📂 Or Pick a Generated Video from Google Drive (Wan2.2, LTX-2-5, MiniMax-H3, etc.)",
                            info="Automatically discovers videos across /content/drive/MyDrive/AEI_3D_Out and AEI_TTS_Out.",
                        )
                        v1_refresh = gr.Button("🔄 Refresh Drive Videos", scale=0)
                    v1_prompt = gr.Textbox(
                        label="Optional Text Prompt (guides specific sound textures alongside visual motion)",
                        lines=2,
                        value="cinematic realistic sound effects matching the visual action, clear high-fidelity foley",
                        info="Leave blank for pure vision-driven Foley, or describe desired materials/environment.",
                    )
                    v1_neg = gr.Textbox(
                        label="Negative Prompt",
                        lines=1,
                        value="music, speech, muffled, low quality, distortion",
                        info="Add 'music' or 'speech' to keep output focused purely on Foley sound effects.",
                    )
                with gr.Column(scale=2):
                    v1_variant = gr.Dropdown(
                        ["large_44k_v2", "large_44k", "medium_44k", "small_44k", "small_16k"],
                        value="large_44k_v2",
                        label="MMAudio Model Variant",
                        info="large_44k_v2 is the highest-quality 44.1 kHz model.",
                    )
                    v1_dur = gr.Slider(1.0, 30.0, value=8.0, step=0.5, label="Max Target Duration (seconds)", info="Automatically truncates to video length if video is shorter.")
                    v1_cfg = gr.Slider(1.0, 10.0, value=4.5, step=0.5, label="CFG Guidance Strength", info="Official default is 4.5.")
                    v1_steps = gr.Slider(10, 50, value=25, step=1, label="Flow-Matching Euler Steps", info="Official default is 25 steps.")
                    v1_seed = gr.Number(value=42, precision=0, label="Seed (-1 = random)", info="Set >=0 for reproducible Foley synthesis.")
                    v1_mask = gr.Checkbox(value=False, label="Mask Away CLIP Semantic Features (Sync-Only Mode)", info="Uses only Synchformer temporal motion cues without CLIP scene semantics.")
                    v1_btn = gr.Button("🎬 Synthesize Synchronized Video Foley", variant="primary")
                    v1_out_vid = gr.Video(label="Synchronized Output Video (.mp4 with 44.1 kHz Audio)")
                    v1_out_aud = gr.Audio(label="Standalone 44.1 kHz Stereo Audio Track (.wav)", type="numpy")
                    with gr.Row():
                        v1_dl_mp4 = gr.File(label="Download Muxed .mp4")
                        v1_dl_wav = gr.File(label="Download Standalone .wav")
                    v1_info = gr.Textbox(label="Run Summary & Drive Paths", lines=4, interactive=False)

            v1_refresh.click(_refresh_videos, outputs=[v1_dr])
            v1_btn.click(
                _ui_v2a,
                inputs=[v1_up, v1_dr, v1_prompt, v1_neg, v1_dur, v1_cfg, v1_steps, v1_seed, v1_mask, v1_variant],
                outputs=[v1_out_vid, v1_out_aud, v1_dl_mp4, v1_dl_wav, v1_info],
            )

        # ── TAB 2: Text-to-Audio Sound Effects (.wav) ──
        with gr.Tab("2. Text-to-Audio Sound Effects & Ambience (.wav)"):
            with gr.Row():
                with gr.Column(scale=3):
                    t2_prompt = gr.Textbox(
                        label="Sound Effect / Ambient Bed Description (Prompt)",
                        lines=3,
                        value=SFX_PRESETS["Rainstorm & Distant Thunder"],
                        info="Describe the acoustic event, materials, environment, and perspective.",
                    )
                    gr.Markdown("**Quick SFX Prompt Presets:**")
                    with gr.Row():
                        for _sname, _sprompt in SFX_PRESETS.items():
                            _b = gr.Button(_sname, size="sm")
                            _b.click(fn=lambda p=_sprompt: p, inputs=None, outputs=[t2_prompt])
                    t2_neg = gr.Textbox(
                        label="Negative Prompt",
                        lines=1,
                        value="music, speech, muffled, low quality",
                        info="Terms to suppress during flow-matching generation.",
                    )
                with gr.Column(scale=2):
                    t2_variant = gr.Dropdown(
                        ["large_44k_v2", "large_44k", "medium_44k", "small_44k", "small_16k"],
                        value="large_44k_v2",
                        label="MMAudio Model Variant",
                        info="large_44k_v2 produces crisp 44.1 kHz sound effects.",
                    )
                    t2_dur = gr.Slider(1.0, 30.0, value=8.0, step=0.5, label="Duration (seconds)", info="8.0s is the native training window; supports 1–30s.")
                    t2_cfg = gr.Slider(1.0, 10.0, value=4.5, step=0.5, label="CFG Guidance Strength", info="Default 4.5.")
                    t2_steps = gr.Slider(10, 50, value=25, step=1, label="Flow-Matching Euler Steps", info="Default 25.")
                    t2_seed = gr.Number(value=42, precision=0, label="Seed (-1 = random)", info="Set >=0 for deterministic sound effects.")
                    t2_btn = gr.Button("🔊 Generate Standalone 44.1 kHz SFX (.wav)", variant="primary")
                    t2_out_aud = gr.Audio(label="Generated Sound Effect (44.1 kHz Stereo)", type="numpy")
                    t2_dl_wav = gr.File(label="Download WAV")
                    t2_info = gr.Textbox(label="SFX Metadata & Drive Paths", lines=4, interactive=False)

            t2_btn.click(
                _ui_t2a,
                inputs=[t2_prompt, t2_neg, t2_dur, t2_cfg, t2_steps, t2_seed, t2_variant],
                outputs=[t2_out_aud, t2_dl_wav, t2_info],
            )

    def _on_load():
        return (
            f"✅ **MMAudio (`{DEFAULT_VARIANT}`) Ready** on `{DEVICE}` · "
            f"Saves muxed `.mp4` and 44.1 kHz `.wav` files to `{OUT_DIR_TTS}` and `{OUT_DIR_3D}`."
        )

    demo.load(_on_load, inputs=None, outputs=[welcome_banner])

clear_output()
demo.queue(concurrency_limit=1).launch(share=True, inline=False, show_error=True)

In [ ]:
# @title STEP 5 — Keep Colab Session Alive
# @markdown Runs a background heartbeat thread to prevent Google Colab from disconnecting during interactive sessions.

import threading, time

def _mmaudio_keepalive():
    while True:
        time.sleep(60)

threading.Thread(target=_mmaudio_keepalive, daemon=True).start()
print("[Keep-Alive] Heartbeat thread active. Outputs:", OUT_DIR_TTS, "|", OUT_DIR_3D)

In [ ]:
# @title STEP 6 — Quick Test (Video-to-Audio or Text-to-Audio without UI)
# @markdown Run a single Video-to-Audio Foley or Text-to-Audio SFX generation from form widgets, play the result inline, and display a `FileLink` download.

MODE = "video_to_audio"  # @param ["video_to_audio", "text_to_audio"]
VIDEO_PATH = ""  # @param {type:"string"}
PROMPT = "Crisp footsteps on gravel and gentle wind blowing through trees"  # @param {type:"string"}
NEGATIVE_PROMPT = "music, speech, muffled"  # @param {type:"string"}
DURATION = 8.0  # @param {type:"slider", min:2.0, max:20.0, step:0.5}
CFG_STRENGTH = 4.5  # @param {type:"slider", min:1.0, max:10.0, step:0.5}
NUM_STEPS = 25  # @param {type:"slider", min:10, max:50, step:1}
SEED = 42  # @param {type:"integer"}
VARIANT = "large_44k_v2"  # @param ["large_44k_v2", "large_44k", "medium_44k", "small_44k", "small_16k"]

import os
from IPython.display import Audio, Video, FileLink, display

if MODE == "video_to_audio":
    cand_vids = discover_aei_drive_videos()
    v_path = VIDEO_PATH.strip() or (cand_vids[0] if cand_vids else os.path.join(CACHE_ROOT, "sample_silent_clip.mp4"))
    if not os.path.exists(v_path):
        raise SystemExit(f"Video not found: {v_path}. Run STEP 2 or switch MODE to 'text_to_audio'.")
    print(f"[QuickTest] Video-to-Audio Foley on {v_path} (variant={VARIANT}) ...")
    res = synthesize_video_foley(
        video_path=v_path,
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        duration=DURATION,
        cfg_strength=CFG_STRENGTH,
        num_steps=NUM_STEPS,
        seed=SEED,
        variant=VARIANT,
        prefix="quicktest_v2a",
    )
    print(f"✓ Saved Muxed MP4 : {res['video_path']}")
    print(f"✓ Saved Stereo WAV: {res['wav_path']} ({res['duration_s']:.2f}s in {res['elapsed_s']:.1f}s)")
    display(Audio(res["wav_path"], autoplay=False))
    display(FileLink(res["video_path"]))
    display(FileLink(res["wav_path"]))
else:
    print(f"[QuickTest] Text-to-Audio SFX (variant={VARIANT}, duration={DURATION}s) ...")
    res = synthesize_text_sfx(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        duration=DURATION,
        cfg_strength=CFG_STRENGTH,
        num_steps=NUM_STEPS,
        seed=SEED,
        variant=VARIANT,
        prefix="quicktest_t2a",
    )
    print(f"✓ Saved Stereo WAV: {res['wav_path']} ({res['duration_s']:.2f}s in {res['elapsed_s']:.1f}s)")
    display(Audio(res["wav_path"], autoplay=False))
    display(FileLink(res["wav_path"]))

In [ ]:
# @title STEP 7 — Batch Foley / Sound Effects Generation with tqdm
# @markdown Processes a batch of video files or text SFX prompts sequentially with `tqdm` and per-item `try`/`except` fault tolerance.

BATCH_MODE = "text_sfx_pack"  # @param ["text_sfx_pack", "video_folder_foley"]
VIDEO_FOLDER_PATH = "/content/drive/MyDrive/AEI_3D_Out/Wan2.2"  # @param {type:"string"}
VIDEO_PROMPT = "realistic synchronized foley sound effects, clear high fidelity audio"  # @param {type:"string"}
DURATION = 8.0  # @param {type:"slider", min:2.0, max:20.0, step:0.5}
CFG_STRENGTH = 4.5  # @param {type:"slider", min:1.0, max:10.0, step:0.5}
NUM_STEPS = 25  # @param {type:"slider", min:10, max:50, step:1}
BASE_SEED = 100  # @param {type:"integer"}
VARIANT = "large_44k_v2"  # @param ["large_44k_v2", "large_44k", "medium_44k", "small_44k", "small_16k"]

import os
from tqdm.auto import tqdm
from IPython.display import Audio, FileLink, display

outputs = []
if BATCH_MODE == "video_folder_foley":
    if os.path.isdir(VIDEO_FOLDER_PATH):
        v_files = sorted(
            os.path.join(VIDEO_FOLDER_PATH, f)
            for f in os.listdir(VIDEO_FOLDER_PATH)
            if f.lower().endswith((".mp4", ".mov", ".webm"))
        )
    else:
        v_files = discover_aei_drive_videos(max_files=5)
    print(f"[Batch] Running Video-to-Audio Foley on {len(v_files)} video(s) ...")
    for idx, vp in enumerate(tqdm(v_files, desc="Step 7 Video Foley Batch")):
        try:
            r = synthesize_video_foley(
                video_path=vp,
                prompt=VIDEO_PROMPT,
                duration=DURATION,
                cfg_strength=CFG_STRENGTH,
                num_steps=NUM_STEPS,
                seed=BASE_SEED + idx,
                variant=VARIANT,
                prefix=f"batch_v2a_{idx+1:03d}",
            )
            outputs.append(r["wav_path"])
            print(f"  [{idx+1:03d}/{len(v_files)}] ✓ {os.path.basename(vp)} -> {r['video_path']} ({r['elapsed_s']:.1f}s)")
        except Exception as e:
            print(f"  [{idx+1:03d}/{len(v_files)}] ✗ FAILED {os.path.basename(vp)}: {e}")
else:
    sfx_prompts = list(SFX_PRESETS.items())
    print(f"[Batch] Generating {len(sfx_prompts)} standalone 44.1 kHz SFX tracks ...")
    for idx, (label, p_text) in enumerate(tqdm(sfx_prompts, desc="Step 7 SFX Pack Batch")):
        try:
            r = synthesize_text_sfx(
                prompt=p_text,
                duration=DURATION,
                cfg_strength=CFG_STRENGTH,
                num_steps=NUM_STEPS,
                seed=BASE_SEED + idx,
                variant=VARIANT,
                prefix=f"batch_sfx_{idx+1:02d}",
            )
            outputs.append(r["wav_path"])
            print(f"  [{idx+1:02d}/{len(sfx_prompts)}] ✓ {label} -> {r['wav_path']} ({r['elapsed_s']:.1f}s)")
        except Exception as e:
            print(f"  [{idx+1:02d}/{len(sfx_prompts)}] ✗ FAILED {label}: {e}")

print(f"\n[Done] {len(outputs)} item(s) saved to {OUT_DIR_TTS} and {OUT_DIR_3D}")
if outputs:
    display(Audio(outputs[-1], autoplay=False))
    display(FileLink(outputs[-1]))